# NVDA Stock Direction Classification

This notebook follows the GeeksforGeeks stock prediction workflow using daily NVDA data from Yahoo Finance. The task is binary classification: predict whether the next trading day's Close is higher than today's. It does not predict a price target or prove a profitable strategy.

Pipeline: download and inspect → EDA → feature engineering and target → chronological holdout split → train-only preprocessing → compare Logistic Regression, polynomial SVC and XGBoost → evaluate.

Reference: [GeeksforGeeks tutorial](https://www.geeksforgeeks.org/machine-learning/stock-price-prediction-using-machine-learning-in-python/). Install if required: pip install yfinance seaborn scikit-learn xgboost.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

## 1. Download and inspect data

Fetch ten years of daily NVDA OHLCV data. Yahoo Finance may return multi-level columns or an incomplete latest row, so normalize columns, sort dates, and remove rows missing essential market values.

In [ ]:
TICKER = "NVDA"
raw = yf.download(TICKER, period="10y", interval="1d", auto_adjust=False,
                  progress=False, threads=False)
if raw.empty:
    raise ValueError("Yahoo Finance returned no rows; check the ticker or try again later.")
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
df = raw.reset_index()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
required = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df = (df[required].dropna(subset=["Date", "Open", "High", "Low", "Close", "Volume"])
      .sort_values("Date").drop_duplicates("Date").reset_index(drop=True))
print(f"{TICKER}: {len(df):,} rows, {df.Date.min().date()} to {df.Date.max().date()}")
display(df.head())
print("Shape:", df.shape)
display(df.describe().T)
df.info()

## 2. Exploratory data analysis

Plot the close price, inspect missingness and compare raw Close with Adj Close. Adjusted prices account for corporate actions; the classifier uses raw OHLC, matching the reference tutorial.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df.Date, df.Close, linewidth=1)
plt.title(f"{TICKER} daily closing price")
plt.xlabel("Date"); plt.ylabel("Price (USD)")
plt.tight_layout(); plt.show()
display(df.isna().sum().to_frame("missing"))
print("Close equals Adj Close:", (df.Close == df["Adj Close"]).sum(), "/", len(df))
factor = df["Adj Close"] / df.Close
print(f"Adjustment factor range: {factor.min():.5f} to {factor.max():.5f}")

In [ ]:
eda_cols = ["Open", "High", "Low", "Close", "Volume"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, eda_cols):
    sns.histplot(df[col], kde=True, ax=ax); ax.set_title(f"{col} distribution")
for ax in axes.flat[len(eda_cols):]: ax.remove()
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, eda_cols):
    sns.boxplot(x=df[col], ax=ax); ax.set_title(f"{col} boxplot")
for ax in axes.flat[len(eda_cols):]: ax.remove()
plt.tight_layout(); plt.show()

## 3. Feature engineering and target

Create calendar features and the tutorial's open-close and low-high features. The last row has no next-day close, so its target is missing and removed instead of being incorrectly labeled 0.

In [ ]:
work = df.copy()
work["day"] = work.Date.dt.day
work["month"] = work.Date.dt.month
work["year"] = work.Date.dt.year
work["is_quarter_end"] = work.month.mod(3).eq(0).astype(int)

yearly = work.groupby("year")[["Open", "High", "Low", "Close"]].mean()
yearly.plot(kind="bar", subplots=True, layout=(2, 2), figsize=(14, 8),
            legend=False, sharex=True)
plt.suptitle("Average OHLC price by year"); plt.tight_layout(); plt.show()
display(work.groupby("is_quarter_end")[["Close", "Volume"]].mean())

work["open-close"] = work.Open - work.Close
work["low-high"] = work.Low - work.High
next_close = work.Close.shift(-1)
work["target"] = np.where(next_close.notna(), (next_close > work.Close).astype(int), np.nan)
work = work.dropna(subset=["target"]).reset_index(drop=True)
work["target"] = work.target.astype(int)

print("Target proportion (1 = next trading day closes higher):")
display(work.target.value_counts(normalize=True).sort_index().rename(
    index={0:"0: flat/down", 1:"1: up"}).to_frame("proportion"))
work.target.value_counts().sort_index().rename(index={0:"0: flat/down", 1:"1: up"}).plot(
    kind="pie", autopct="%1.1f%%", ylabel="", title="Next-day direction")
plt.show()

In [ ]:
corr_cols = ["Open", "High", "Low", "Close", "Volume", "open-close",
             "low-high", "is_quarter_end", "target"]
plt.figure(figsize=(10, 8))
sns.heatmap(work[corr_cols].corr(numeric_only=True), cmap="coolwarm",
            center=0, annot=True, fmt=".2f")
plt.title("Feature correlation matrix"); plt.tight_layout(); plt.show()

## 4. Chronological split and preprocessing

A random split can put adjacent trading days in both sets and overstate performance. Reserve the latest 10% as a holdout. Preprocessing is placed inside each model pipeline, so imputation/scaling parameters are learned from training data only.

In [ ]:
FEATURES = ["open-close", "low-high", "is_quarter_end"]
split_idx = int(len(work) * 0.90)
train, valid = work.iloc[:split_idx].copy(), work.iloc[split_idx:].copy()
X_train, y_train = train[FEATURES], train.target
X_valid, y_valid = valid[FEATURES], valid.target
print(f"Train: {len(train)} rows, {train.Date.min().date()} to {train.Date.max().date()}")
print(f"Holdout: {len(valid)} rows, {valid.Date.min().date()} to {valid.Date.max().date()}")
print("Shapes:", X_train.shape, X_valid.shape)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Majority-class baseline accuracy: {accuracy_score(y_valid, baseline.predict(X_valid)):.3f}")

## 5. Model development and evaluation

Compare Logistic Regression, polynomial-kernel SVC and XGBoost as in the reference. Report ROC-AUC as well as threshold-based metrics. A train/holdout gap can indicate overfitting.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ]),
    "Polynomial SVC": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="poly", probability=True, class_weight="balanced", random_state=42)),
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            eval_metric="logloss", random_state=42, n_jobs=-1)),
    ]),
}

def score_model(model, X_fit, y_fit, X_eval, y_eval):
    model.fit(X_fit, y_fit)
    p_fit = model.predict_proba(X_fit)[:, 1]
    p_eval = model.predict_proba(X_eval)[:, 1]
    pred = (p_eval >= 0.5).astype(int)
    return {
        "train_roc_auc": roc_auc_score(y_fit, p_fit),
        "holdout_roc_auc": roc_auc_score(y_eval, p_eval),
        "holdout_accuracy": accuracy_score(y_eval, pred),
        "holdout_balanced_accuracy": balanced_accuracy_score(y_eval, pred),
        "holdout_precision": precision_score(y_eval, pred, zero_division=0),
        "holdout_recall": recall_score(y_eval, pred, zero_division=0),
        "holdout_f1": f1_score(y_eval, pred, zero_division=0),
    }

results = {name: score_model(model, X_train, y_train, X_valid, y_valid)
           for name, model in models.items()}
results_df = pd.DataFrame(results).T.sort_values("holdout_roc_auc", ascending=False)
display(results_df.round(3))
best_name = results_df.index[0]
best_model = models[best_name]
print("Best by holdout ROC-AUC:", best_name)

## 6. Confusion matrix and conclusion

The confusion matrix evaluates the selected model at a 0.5 probability threshold. Treat this holdout as a final evaluation set. A score near 0.5 ROC-AUC or near the simple baseline suggests little predictive signal in this experiment.

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    best_model, X_valid, y_valid, display_labels=["Down / not up", "Up"])
plt.title(f"{best_name}: holdout confusion matrix")
plt.tight_layout(); plt.show()
print("Holdout:", valid.Date.min().date(), "to", valid.Date.max().date())
print("This is a classification experiment, not a backtest. It omits transaction costs, execution timing, position sizing, and risk management.")

## Summary

This project follows the main GeeksforGeeks sequence with NVDA data and adds time-ordered evaluation, training-only preprocessing, a majority-class baseline, and more evaluation metrics. It predicts next-day direction, not price. Results should not be interpreted as investment returns.

Data source: Yahoo Finance through the yfinance package. Data availability and historical values can change.